# ADA — Official Colab Pretraining Notebook

**This is the only supported Colab entry point for ADA pretraining.** It streams real FineWeb-Edu records, trains ADA from random initialization, evaluates on a held-out stream, and saves resumable checkpoints and metrics to Google Drive.

This is a bounded research pilot, not frontier-scale training. The starter corpus is English-only; do not interpret this run as multilingual or specialist capability.

**Run order:** GPU check → clone current `main` → install dependencies → mount Drive → inspect real records → train → inspect loss and checkpoint.

In [ ]:
# 1. Runtime check — in Colab choose Runtime > Change runtime type > T4 GPU (or better)
import torch, os, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. In Colab select Runtime > Change runtime type > GPU, then rerun this cell.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
# 2. Get the latest ADA main branch and install the one required training extra
import os, pathlib, subprocess, sys
REPO_DIR = pathlib.Path('/content/ADA')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/vimalspaceton618-afk/ADA.git', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[pretraining]'], cwd=REPO_DIR, check=True)
print('ADA source:', REPO_DIR)
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_DIR, text=True).strip())

In [ ]:
# 3. Mount Google Drive — all training artifacts are written here, not to ephemeral /content
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/ADA_Pretraining')
RUN_DIR = DRIVE_ROOT / 'fineweb_edu_pilot'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Persistent run directory:', RUN_DIR)
print('Existing files:', [p.name for p in RUN_DIR.iterdir()])

In [ ]:
# 4. Inspect real training-corpus records before starting
from datasets import load_dataset
from pprint import pprint
preview = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True, revision='87f09149ef4734204d70ed1d046ddc9ca3f2b8f9')
for i, row in enumerate(preview.take(3), start=1):
    print(f'--- RECORD {i} ---')
    pprint({k: v for k, v in row.items() if k != 'text'})
    print((row.get('text') or '')[:1200], '\\n')

## Training configuration

The defaults below run **500 optimizer steps × batch 4 × sequence length 256 = 512,000 next-token targets**. Start with these settings; increase only after checking GPU memory, throughput, and validation-loss behavior. The script trains a tokenizer on training-only documents and creates a deterministic held-out validation stream.

On a fresh run keep `RESUME = False`. If Colab disconnects after a checkpoint is saved, set `RESUME = True` and rerun the training cell. Do not change the model/tokenizer settings when resuming.

In [ ]:
# 5. Launch actual real-data pretraining (not a synthetic smoke test)
MAX_STEPS = 500
BATCH_SIZE = 4
SEQUENCE_LENGTH = 256
VOCAB_SIZE = 8192
TOKENIZER_DOCS = 20000
RESUME = False  # Change to True only to continue this exact run from its saved checkpoint
args = [
    sys.executable, 'scripts/ada_pretrain_fineweb_edu.py',
    '--device', 'cuda', '--output-dir', str(RUN_DIR),
    '--max-steps', str(MAX_STEPS), '--batch-size', str(BATCH_SIZE),
    '--sequence-length', str(SEQUENCE_LENGTH), '--vocab-size', str(VOCAB_SIZE),
    '--tokenizer-docs', str(TOKENIZER_DOCS), '--eval-every', '100',
    '--eval-batches', '8', '--save-every', '50'
]
if RESUME:
    args.append('--resume')
print('Command:', ' '.join(args))
subprocess.run(args, cwd=REPO_DIR, check=True)

In [ ]:
# 6. Plot the actual training and held-out validation metrics
import json, pandas as pd, matplotlib.pyplot as plt
metrics_path = RUN_DIR / 'metrics.jsonl'
if not metrics_path.exists():
    raise FileNotFoundError(f'Metrics not found: {metrics_path}')
rows = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
metrics = pd.DataFrame(rows)
display(metrics.tail(10))
plt.figure(figsize=(10, 5))
if 'train_loss' in metrics:
    train = metrics.dropna(subset=['train_loss'])
    if len(train): plt.plot(train['step'], train['train_loss'], marker='o', label='Train loss')
valid = metrics.dropna(subset=['validation_loss'])
if len(valid): plt.plot(valid['step'], valid['validation_loss'], marker='s', label='Validation loss')
plt.xlabel('Optimizer step'); plt.ylabel('Next-token cross-entropy'); plt.title('ADA FineWeb-Edu pretraining')
plt.grid(True, alpha=0.25); plt.legend(); plt.show()

In [ ]:
# 7. Verify the saved run summary, manifest, tokenizer and model checkpoint
for name in ['run_summary.json', 'run_manifest.json']:
    path = RUN_DIR / name
    print(f'\\n===== {name} =====')
    if path.exists(): print(path.read_text())
    else: print('Not written yet — training may have been interrupted before completion.')
required = [RUN_DIR / 'tokenizer.json', RUN_DIR / 'model' / 'config.json', RUN_DIR / 'model' / 'pytorch_model.bin', RUN_DIR / 'training_state.pt', metrics_path]
for path in required:
    print(('OK  ' if path.exists() else 'MISS'), path, (f'{path.stat().st_size:,} bytes' if path.exists() and path.is_file() else ''))
print('Drive run folder:', RUN_DIR)

## What this run does — and does not — establish

- It trains ADA weights from random initialization on streamed, real FineWeb-Edu records.
- It records training/validation loss, tokens seen, run configuration, and resumable checkpoints.
- FineWeb-Edu is English educational web text. It is not an India-language mixture, code corpus, cybersecurity corpus, or balanced domain dataset.
- A falling loss is evidence of optimization on this corpus, not proof of reasoning, safety, general intelligence, or frontier capability.
- Review the dataset card and applicable terms before redistribution: https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu

**Next gate:** inspect the completed `run_summary.json`, validation curve, tokens/sec, and GPU memory before selecting a larger run or adding multilingual/domain data.